In [ ]:
import os, time, asyncio
import nest_asyncio
nest_asyncio.apply()   # lets readframe re-enter the kernel loop
import io
import numpy as np
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, HTML

from pynq import Overlay
from pynq.lib.video import VideoMode

BIT_PATH = '/home/xilinx/jupyter_notebooks/wave_simulator_demo/v18.bit'

SIM_DIV = 2
DISP_W, DISP_H = 640, 480
SIM_W, SIM_H = DISP_W // SIM_DIV, DISP_H // SIM_DIV   # 320 x 240



In [ ]:
# load the bitstream and bring up the video pipeline
overlay    = Overlay(BIT_PATH)
pixgen     = overlay.pixel_generator_0
imgen_vdma = overlay.video.axi_vdma_0.readchannel
hdmi_out   = overlay.video.hdmi_out
hdmi_out._vdma = overlay.video.axi_vdma   # this overlay family needs the vdma handle set by hand

# clean slate in case a previous run left the pipeline up
for fn in [imgen_vdma.stop, hdmi_out.close]:
    try:
        fn(); time.sleep(0.25)
    except Exception as e:
        print('  skip:', repr(e))

vmode = VideoMode(DISP_W, DISP_H, 32)   # 32bpp has to match the hdl
imgen_vdma.mode = vmode
imgen_vdma.start()
time.sleep(0.4)
hdmi_out.configure(vmode)
hdmi_out.start()
time.sleep(0.4)



In [ ]:
# v17 register map (checked against pixel_generator_320x240.v + eden/jerry notes)
#   gp0  bit0  fire one burst (rising edge)
#   gp1  (y<<10)|x   source position in display coords, hdl divides by SIM_DIV
#   gp2  bit31 enable | (y<<10)|x   object centre in display coords
#   gp3  (half_h<<8)|half_w   object size in SIM cells (0 = hdl default 10)
#   gp4  [8:0] source gain / amplitude
#   gp5  [15:8] source duration in frames (how long the burst injects)
# gp6/gp7 are not used by this build, so don't touch them.

def _cl(v, lo, hi):
    return max(lo, min(hi, int(v)))

def pack_xy(x, y):
    return (_cl(y, 0, DISP_H-1) << 10) | _cl(x, 0, DISP_W-1)

def configure_wave_source(x, y, gain=160, duration=16):
    pixgen.register_map.gp1 = pack_xy(x, y)
    pixgen.register_map.gp4 = _cl(gain, 0, 255)
    pixgen.register_map.gp5 = (_cl(duration, 1, 255) & 0xFF) << 8

def configure_object(x, y, half_w=15, half_h=15, enable=True):
    # object only moves when enable is set, otherwise the hdl pins it to the
    # default centred square. half_w/half_h are in sim cells, min 1.
    ctrl = (1 << 31) if enable else 0
    pixgen.register_map.gp2 = ctrl | pack_xy(x, y)
    pixgen.register_map.gp3 = ((_cl(half_h, 1, SIM_H//2) & 0xFF) << 8) | (_cl(half_w, 1, SIM_W//2) & 0xFF)

def fire_source():
    # while the anim runs, flag the request and let the frame loop make the
    # gp0[0] rising edge, so spamming space does not stall the display
    global _fire_req
    if _anim_on:
        _fire_req = True
    else:
        pixgen.register_map.gp0 = 0; time.sleep(0.01)
        pixgen.register_map.gp0 = 1; time.sleep(0.01)
        pixgen.register_map.gp0 = 0

def clear_field():
    pixgen.register_map.gp0 = 0; time.sleep(0.01)
    pixgen.register_map.gp0 = 2; time.sleep(0.04)
    pixgen.register_map.gp0 = 0



In [ ]:
# run the frame loop as an async task on the kernel's own loop so the buttons
# and keys still respond between frames. the plain readframe/writeframe block
# the loop, so we use pynq's async versions and await them instead
_anim_on = False
_fire_req = False
_fire_hold = 0
_snap_req = False
_wvdma = overlay.video.axi_vdma
_wchan = getattr(_wvdma, 'writechannel', None)

async def _read_frame():
    if hasattr(imgen_vdma, 'readframe_async'):
        return await imgen_vdma.readframe_async()
    return imgen_vdma.readframe()

async def _write_frame(frame):
    if hasattr(hdmi_out, 'writeframe_async'):
        await hdmi_out.writeframe_async(frame)
    elif _wchan is not None and hasattr(_wchan, 'writeframe_async'):
        await _wchan.writeframe_async(frame)
    else:
        hdmi_out.writeframe(frame)

async def _anim():
    global _anim_on, _fire_req, _fire_hold, _snap_req
    n = 0
    while _anim_on:
        # make a clean gp0[0] rising edge over 2 frames, no blocking sleeps
        if _fire_hold > 0:
            _fire_hold -= 1
            if _fire_hold == 0:
                pixgen.register_map.gp0 = 0
        elif _fire_req:
            _fire_req = False
            pixgen.register_map.gp0 = 1
            _fire_hold = 2
        try:
            frame = await _read_frame()
            await _write_frame(frame)
        except Exception as e:
            print('[anim]', repr(e)); _anim_on = False; break
        if _snap_req:
            _snap_req = False
            _save_snapshot(frame)
        n += 1
        if st.get('autofire') and n % 90 == 0:   # auto sweep ~ every 1.5s
            _fire_req = True
        await asyncio.sleep(0)

def start_anim():
    global _anim_on
    if _anim_on: return
    _anim_on = True
    asyncio.ensure_future(_anim())

def stop_anim():
    global _anim_on
    _anim_on = False


In [ ]:
# wave simulator ui, voila version
from IPython.display import HTML, display
Layout, HBox, VBox = widgets.Layout, widgets.HBox, widgets.VBox

# dark theme for the whole voila page + the widgets
display(HTML(r"""
<style>
  body, #rendered_cells, .jp-Notebook, .jp-NotebookPanel { background:#0d1117 !important; }
  .jp-Cell, .jp-OutputArea, .jp-OutputArea-output, .jp-RenderedHTMLCommon,
  .cell, .output_subarea, .rendered_html, .jp-MarkdownOutput { background:transparent !important; box-shadow:none !important; }
  .jp-OutputArea-prompt, .jp-InputPrompt, .prompt { display:none !important; }
  .widget-label, .jupyter-widgets, .widget-html-content { color:#e6edf3 !important; }
  .jupyter-button { background:#161b22 !important; color:#e6edf3 !important; border:1px solid #30363d !important;
     border-radius:6px !important; font-family:Menlo,Consolas,monospace !important; }
  .jupyter-button:hover { border-color:#58a6ff !important; }
  .jupyter-button.mod-success { background:#238636 !important; color:#fff !important; border-color:#2ea043 !important; }
  .jupyter-button.mod-danger  { background:#da3633 !important; color:#fff !important; }
  .jupyter-button.mod-warning { background:#9e6a03 !important; color:#fff !important; }
  .jupyter-button.mod-info    { background:#1f6feb !important; color:#fff !important; }
  #sv-ctrl table, #sv-ctrl thead, #sv-ctrl tbody, #sv-ctrl tr, #sv-ctrl td, #sv-ctrl th {
     background:transparent !important; border:none !important; color:#e6edf3 !important; }
</style>
"""))

# animated wave ripples in the page background
display(HTML(r"""
<div class="sv-bg"><span></span><span></span><span></span><span></span><span></span></div>
<style>
  body { background:#0a0e14 !important; }
  .sv-bg{ position:fixed; inset:0; z-index:-1; pointer-events:none; overflow:hidden;
          background:radial-gradient(circle at 50% 50%, #0f1a2a 0%, #0a0e14 72%); }
  /* fixed-size circle scaled on the GPU - no per-frame layout/paint, so it
     stays smooth at full screen. animating width/height was the lag cause. */
  .sv-bg span{ position:absolute; left:50%; top:50%; width:1800px; height:1800px;
          margin:-900px 0 0 -900px;
          border:2px solid rgba(63,185,80,0.42); border-radius:50%;
          opacity:0; transform:scale(0); will-change:transform, opacity;
          animation:sv-ping 6s linear infinite; }
  .sv-bg span:nth-child(2){ animation-delay:1.2s; }
  .sv-bg span:nth-child(3){ animation-delay:2.4s; }
  .sv-bg span:nth-child(4){ animation-delay:3.6s; }
  .sv-bg span:nth-child(5){ animation-delay:4.8s; }
  @keyframes sv-ping{ 0%{ transform:scale(0.004); opacity:0; }
                      8%{ opacity:.75; }
                      100%{ transform:scale(1); opacity:0; } }
</style>
"""))

# controller state
STEP = 20
st = dict(src_x=160, src_y=240, obj_x=400, obj_y=240,
          obj_hw=15, obj_hh=15, gain=160, duration=16, obj_on=True, autofire=False)

def push_src(): configure_wave_source(st['src_x'], st['src_y'], st['gain'], st['duration'])
def push_obj(): configure_object(st['obj_x'], st['obj_y'], st['obj_hw'], st['obj_hh'], st['obj_on'])
def reset_scene():
    st.update(src_x=160, src_y=240, obj_x=400, obj_y=240,
              obj_hw=15, obj_hh=15, gain=160, duration=16, obj_on=True)
    push_src(); push_obj(); clear_field()

status_html = widgets.HTML()

def refresh():
    sx, sy = st['src_x'], st['src_y']; ox, oy = st['obj_x'], st['obj_y']
    status_html.value = (
      "<div style='font-family:Menlo,Consolas,monospace;font-size:12px;color:#e6edf3;"
      "background:rgba(13,17,23,0.80);border:1px solid #30363d;border-radius:10px;padding:14px 16px;min-width:230px;'>"
      "<div style='color:#3fb950;font-weight:bold;margin-bottom:8px;'>live values</div>"
      "<div style='line-height:1.9'>source&nbsp;&nbsp;<span style='color:#79c0ff'>({},{})</span> &nbsp;sim ({},{})</div>"
      "<div style='line-height:1.9'>object&nbsp;&nbsp;<span style='color:#79c0ff'>({},{})</span> &nbsp;sim ({},{}) &nbsp;<b>{}</b></div>"
      "<div style='line-height:1.9'>size&nbsp;&nbsp;&nbsp;&nbsp;w=<span style='color:#79c0ff'>{}</span> &nbsp;h=<span style='color:#79c0ff'>{}</span></div>"
      "<div style='line-height:1.9'>gain&nbsp;&nbsp;&nbsp;&nbsp;<span style='color:#79c0ff'>{}</span> &nbsp;pulse <span style='color:#79c0ff'>{}</span></div>"
      "<div style='line-height:1.9'>anim&nbsp;&nbsp;&nbsp;&nbsp;<span style='color:#3fb950'>{}</span></div>"
      "</div>"
    ).format(sx,sy,sx//SIM_DIV,sy//SIM_DIV, ox,oy,ox//SIM_DIV,oy//SIM_DIV,
             'ON' if st['obj_on'] else 'off', st['obj_hw'], st['obj_hh'],
             st['gain'], st['duration'], 'ON' if _anim_on else 'off')

def on_key(k):
    if   k == 'space': fire_source()
    elif k == 'w': st['src_y'] = _cl(st['src_y']-STEP, 0, DISP_H-1); push_src()
    elif k == 's': st['src_y'] = _cl(st['src_y']+STEP, 0, DISP_H-1); push_src()
    elif k == 'a': st['src_x'] = _cl(st['src_x']-STEP, 0, DISP_W-1); push_src()
    elif k == 'd': st['src_x'] = _cl(st['src_x']+STEP, 0, DISP_W-1); push_src()
    elif k == 'ArrowUp':    st['obj_y'] = _cl(st['obj_y']-STEP, 0, DISP_H-1); push_obj()
    elif k == 'ArrowDown':  st['obj_y'] = _cl(st['obj_y']+STEP, 0, DISP_H-1); push_obj()
    elif k == 'ArrowLeft':  st['obj_x'] = _cl(st['obj_x']-STEP, 0, DISP_W-1); push_obj()
    elif k == 'ArrowRight': st['obj_x'] = _cl(st['obj_x']+STEP, 0, DISP_W-1); push_obj()
    elif k == 'g': st['gain'] = _cl(st['gain']+10, 0, 255); push_src()
    elif k == 'f': st['gain'] = _cl(st['gain']-10, 0, 255); push_src()
    elif k == ']': st['duration'] = _cl(st['duration']+2, 1, 255); push_src()
    elif k == '[': st['duration'] = _cl(st['duration']-2, 1, 255); push_src()
    elif k == 'e': st['obj_on'] = not st['obj_on']; push_obj()
    elif k == 'r': reset_scene()
    refresh()

def adj_size(field, d):
    if   field == 'w': st['obj_hw'] = _cl(st['obj_hw']+d, 1, SIM_W//2)
    elif field == 'h': st['obj_hh'] = _cl(st['obj_hh']+d, 1, SIM_H//2)
    push_obj(); refresh()
def adj_both(d):
    st['obj_hw'] = _cl(st['obj_hw']+d, 1, SIM_W//2)
    st['obj_hh'] = _cl(st['obj_hh']+d, 1, SIM_H//2)
    push_obj(); refresh()

# snapshot: save the current fpga frame as a full-res png for the report
snap_view = widgets.Image(format='png',
    layout=Layout(max_width='340px', margin='6px', border='1px solid #30363d'))
snap_view.layout.display = 'none'
snap_lbl = widgets.HTML()

def _save_snapshot(frame):
    arr = np.array(frame)
    img = Image.fromarray(arr[:, :, :3], 'RGB')
    os.makedirs('captures', exist_ok=True)
    name = 'captures/wave_' + time.strftime('%Y%m%d_%H%M%S') + '.png'
    img.save(name)
    buf = io.BytesIO(); img.save(buf, format='png')
    snap_view.value = buf.getvalue()
    snap_view.layout.display = ''
    snap_lbl.value = "<span style='color:#3fb950;font-family:Menlo;font-size:12px'>saved " + name + "</span>"

def request_snapshot():
    global _snap_req
    if not _anim_on:
        snap_lbl.value = "<span style='color:#d29922;font-family:Menlo;font-size:12px'>press Launch first</span>"
        return
    _snap_req = True

def shutdown():
    stop_anim(); time.sleep(0.3)
    try: pixgen.register_map.gp0 = 0
    except Exception: pass
    for fn in [imgen_vdma.stop, hdmi_out.close]:
        try: fn(); time.sleep(0.2)
        except Exception: pass
    control_box.layout.display = 'none'
    home_area.children = [stopped_lbl, restart_btn]
    home_area.layout.display = ''

def restart_pipeline():
    # bitstream is still loaded after shutdown, so just bring the video
    # pipeline and animation back up in this same kernel
    global imgen_vdma, hdmi_out, _wvdma, _wchan
    imgen_vdma = overlay.video.axi_vdma_0.readchannel
    hdmi_out   = overlay.video.hdmi_out
    hdmi_out._vdma = overlay.video.axi_vdma
    _wvdma = overlay.video.axi_vdma
    _wchan = getattr(_wvdma, 'writechannel', None)
    for fn in [imgen_vdma.stop, hdmi_out.close]:
        try: fn(); time.sleep(0.2)
        except Exception: pass
    imgen_vdma.mode = vmode
    imgen_vdma.start(); time.sleep(0.4)
    hdmi_out.configure(vmode); hdmi_out.start(); time.sleep(0.4)
    home_area.layout.display = 'none'
    control_box.layout.display = ''
    push_src(); push_obj(); start_anim(); refresh()

# buttons. each one gets a css class so the keyboard handler can click it
def _btn(label,k,cls,w='52px'):
    b=widgets.Button(description=label, layout=Layout(width=w, margin='3px')); b.add_class(cls)
    b.on_click(lambda _e, kk=k: on_key(kk)); return b
def _act(label,cls,fn,style='',w='72px'):
    b=widgets.Button(description=label, button_style=style, layout=Layout(width=w, margin='3px'))
    if cls: b.add_class(cls)
    b.on_click(lambda _e: fn()); return b

fire_b  = _act('FIRE','sv-fire', lambda: on_key('space'), 'success','110px')
start_b = _act('start','', lambda:(start_anim(),refresh()), 'info','72px')
stop_b  = _act('pause','', lambda:(stop_anim(),refresh()), '','72px')
sweep_chk = widgets.Checkbox(value=False, description='auto sweep', indent=False, layout=Layout(width='120px', margin='3px'))
sweep_chk.observe(lambda c: st.update(autofire=bool(c['new'])), names='value')

src_pad = VBox([
    HBox([_btn('W','w','sv-w')], layout=Layout(justify_content='center')),
    HBox([_btn('A','a','sv-a'), _btn('S','s','sv-s'), _btn('D','d','sv-d')]),
], layout=Layout(border='1px solid #21262d', border_radius='8px', padding='6px', margin='4px'))
obj_pad = VBox([
    HBox([_btn('Up','ArrowUp','sv-up')], layout=Layout(justify_content='center')),
    HBox([_btn('Lf','ArrowLeft','sv-left'), _btn('Dn','ArrowDown','sv-down'), _btn('Rt','ArrowRight','sv-right')]),
], layout=Layout(border='1px solid #21262d', border_radius='8px', padding='6px', margin='4px'))

w_minus=_act('W -','sv-wm',lambda:adj_size('w',-1),'','52px'); w_plus=_act('W +','sv-wp',lambda:adj_size('w',1),'','52px')
h_minus=_act('H -','sv-hm',lambda:adj_size('h',-1),'','52px'); h_plus=_act('H +','sv-hp',lambda:adj_size('h',1),'','52px')
size_minus=_act('size -','sv-sm',lambda:adj_both(-1),'','70px'); size_plus=_act('size +','sv-sp',lambda:adj_both(1),'','70px')
gain_minus=_act('gain -','sv-gm',lambda:on_key('f'),'','70px'); gain_plus=_act('gain +','sv-gp',lambda:on_key('g'),'','70px')
dur_minus=_act('pulse -','sv-dm',lambda:on_key('['),'','80px'); dur_plus=_act('pulse +','sv-dp',lambda:on_key(']'),'','80px')
obj_toggle=_act('object on/off','sv-e',lambda:on_key('e'),'','124px')

reset_b=_act('reset','sv-r',lambda:on_key('r'),'','80px')
snap_b =_act('snapshot','',request_snapshot,'info','100px')
shut_b =_act('shutdown','',shutdown,'danger','100px')

buttons = VBox([
    HBox([fire_b, start_b, stop_b, sweep_chk]),
    HBox([widgets.HTML("<span style='color:#8b949e;font-family:Menlo;font-size:11px;margin:0 6px'>move</span>"), src_pad, obj_pad],
         layout=Layout(align_items='center')),
    HBox([widgets.Label('size'), w_minus, w_plus, h_minus, h_plus, size_minus, size_plus], layout=Layout(align_items='center')),
    HBox([gain_minus, gain_plus, dur_minus, dur_plus, obj_toggle]),
    HBox([reset_b, snap_b, shut_b]),
], layout=Layout(padding='6px'))

# buttons on the left, live values right beside them
control_inner = HBox([buttons, status_html], layout=Layout(align_items='flex-start', justify_content='center'))
control_box = VBox([control_inner, snap_lbl, snap_view], layout=Layout(margin='6px 0', align_items='center'))
control_box.layout.display = 'none'

# keyboard + legend panel. raw html so the script actually runs
js_panel = HTML(r"""
<div id="sv-wrap" style="display:flex;justify-content:center;">
<div id="sv-ctrl" tabindex="0"
     style="width:470px;padding:14px 18px;margin:10px auto;
            background:rgba(13,17,23,0.80);border:2px solid #30a46c;border-radius:10px;
            font-family:Menlo,Consolas,monospace;color:#e6edf3;outline:none;user-select:none;cursor:default;">
  <div style="color:#3fb950;font-weight:bold;margin-bottom:8px;">
    Controller
    <span id="sv-badge" style="float:right;font-size:11px;font-weight:normal;color:#8b949e;
          background:#161b22;padding:2px 8px;border-radius:4px;">click to activate</span>
  </div>
  <table style="font-size:12px;line-height:1.85;border-collapse:collapse;width:100%;">
    <tr><td style="color:#79c0ff;width:160px">space</td><td>fire pulse</td></tr>
    <tr><td style="color:#79c0ff">W A S D</td><td>move source</td></tr>
    <tr><td style="color:#79c0ff">arrows</td><td>move object</td></tr>
    <tr><td style="color:#79c0ff">scroll</td><td>resize (w + h)</td></tr>
    <tr><td style="color:#79c0ff">shift / ctrl scroll</td><td>width / height only</td></tr>
    <tr><td style="color:#79c0ff">g / f</td><td>gain up / down</td></tr>
    <tr><td style="color:#79c0ff">] / [</td><td>pulse length up / down</td></tr>
    <tr><td style="color:#79c0ff">e / r</td><td>object mine/default / reset</td></tr>
  </table>
  <div style="margin-top:8px;font-size:10px;color:#484f58;">click panel to toggle keys (blue = ON). buttons always work.</div>
</div>
</div>
<script>
(function(){
  var armed=false;
  function click(cls){ var el=document.querySelector('.'+cls)||document.querySelector('.'+cls+' button'); if(el){el.click();return true;} return false; }
  var KMAP={' ':'sv-fire','w':'sv-w','a':'sv-a','s':'sv-s','d':'sv-d','ArrowUp':'sv-up','ArrowDown':'sv-down',
    'ArrowLeft':'sv-left','ArrowRight':'sv-right','g':'sv-gp','f':'sv-gm',']':'sv-dp','[':'sv-dm','e':'sv-e','r':'sv-r'};
  function onKey(ev){
    if(!armed) return;
    var tag=((ev.target&&ev.target.tagName)||'').toLowerCase();
    if(tag==='input'||tag==='textarea') return;
    var cls=KMAP[ev.key]; if(!cls) return;
    ev.preventDefault(); ev.stopPropagation(); click(cls);
  }
  document.addEventListener('keydown', onKey, true);
  function setKM(on){ try{ if(window.Jupyter){ on?Jupyter.keyboard_manager.enable():Jupyter.keyboard_manager.disable(); } }catch(e){} }
  function setup(){
    var el=document.getElementById('sv-ctrl'); var badge=document.getElementById('sv-badge');
    if(!el){ setTimeout(setup,200); return; }
    function arm(on){ armed=on; el.style.borderColor=on?'#58a6ff':'#30a46c';
      badge.textContent=on?'keys ON':'click to activate'; badge.style.color=on?'#3fb950':'#8b949e'; setKM(!on); }
    el.addEventListener('click', function(ev){ ev.stopPropagation(); arm(!armed); });
    el.addEventListener('wheel', function(ev){ ev.preventDefault();
      var up=(ev.deltaY!==0)?(ev.deltaY<0):(ev.deltaX<0);
      if(ev.shiftKey) click(up?'sv-wp':'sv-wm'); else if(ev.ctrlKey) click(up?'sv-hp':'sv-hm'); else click(up?'sv-sp':'sv-sm');
    }, {passive:false});
    arm(false);
  }
  setup();
})();
</script>
""")

# launch screen and control screen
title_w = widgets.HTML("<div style='text-align:center;width:100%;'>"
   "<div style='font-size:42px;font-weight:bold;color:#3fb950;font-family:Menlo,Consolas,monospace;"
   "letter-spacing:2px;padding:24px 0 6px;'>Wave Simulator</div></div>")

launch_btn  = widgets.Button(description='Launch Demo', button_style='success', icon='play',
                             layout=Layout(width='220px', height='48px', margin='10px'))
restart_btn = widgets.Button(description='Restart', button_style='success',
                             layout=Layout(width='220px', height='48px', margin='10px'))
restart_btn.on_click(lambda _e: restart_pipeline())
stopped_lbl = widgets.HTML("<div style='color:#8b949e;text-align:center;font-family:Menlo,Consolas,monospace;'>stopped - press Restart to resume</div>")

home_area = VBox([launch_btn], layout=Layout(align_items='center', margin='6px 0'))

def start_demo(_b=None):
    home_area.layout.display = 'none'
    control_box.layout.display = ''
    push_src(); push_obj(); start_anim(); refresh()
launch_btn.on_click(start_demo)

refresh()
display(title_w)
display(home_area)
display(js_panel)
display(control_box)
